# Existing Code Reuse — Search Feature Generation

This notebook consumes the static capability catalog produced by notebook 01, or creates a small AST-only fallback catalog from the installed packaging distribution. It never imports or executes cataloged target packages.

The notebook builds a long-form, append-only feature registry. Attributes, labels, tags, phrases, ports, graph edges, onion layers, blocking keys, and embeddings all use the same extensible row schema and retain generator, revision, evidence, configuration, and model provenance.

Default settings are intentionally modest for a Kaggle CPU session. Increase the scale knobs only after recording a baseline run.


In [ ]:
# Lightweight retrieval dependencies. The semantic encoder library is installed, but model
# weights are downloaded only if a sentence-transformer model is explicitly selected below.
!pip install -q "packaging>=24,<27" "numpy>=1.26,<3" "pandas>=2,<3" "pyarrow>=15,<22" "scikit-learn>=1.4,<2" "datasketch>=1.6,<2" "networkx>=3.2,<4" "sentence-transformers>=3,<6"


In [ ]:
from __future__ import annotations

import ast
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import re
import sqlite3
import subprocess
import sys
import time
import unicodedata
from typing import Any, Iterable, Mapping, Sequence

from datasketch import MinHash
import networkx as nx
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import HashingVectorizer, TfidfVectorizer


def canonical_json(value: Any) -> str:
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def digest_json(value: Any) -> str:
    return "sha256:" + hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return "sha256:" + digest.hexdigest()


IS_KAGGLE = Path("/kaggle/working").is_dir()
OUTPUT_ROOT = Path(
    os.environ.get(
        "REUSE_OUTPUT_ROOT",
        "/kaggle/working" if IS_KAGGLE else "./kaggle_working",
    )
)
CATALOG_HINT = os.environ.get("REUSE_CATALOG_PATH", "").strip()

MODEL_REGISTRY: dict[str, dict[str, Any]] = {
    "hashing-word-384-v1": {
        "backend": "sklearn_hashing",
        "model_id": "sklearn.feature_extraction.text.HashingVectorizer",
        "revision": f"sklearn-{sklearn.__version__}:word-384-v1",
        "dimension": 384,
        "analyzer": "word",
        "ngram_range": [1, 2],
        "alternate_sign": False,
    },
    "hashing-char-512-v1": {
        "backend": "sklearn_hashing",
        "model_id": "sklearn.feature_extraction.text.HashingVectorizer",
        "revision": f"sklearn-{sklearn.__version__}:char-wb-512-v1",
        "dimension": 512,
        "analyzer": "char_wb",
        "ngram_range": [3, 5],
        "alternate_sign": False,
    },
    "all-minilm-l6-v2-pinned": {
        "backend": "sentence_transformers",
        "model_id": "sentence-transformers/all-MiniLM-L6-v2",
        "revision": os.environ.get(
            "MINILM_REVISION",
            "c9745ed1d9f207416be6d2e6f8de32d1f16199bf",
        ),
        "dimension": 384,
        "batch_size": 64,
    },
}

USE_SEMANTIC_MODEL = os.environ.get("USE_SENTENCE_TRANSFORMER", "0") == "1"
DOCUMENTATION_MODEL = (
    "all-minilm-l6-v2-pinned" if USE_SEMANTIC_MODEL else "hashing-word-384-v1"
)

CONFIG: dict[str, Any] = {
    "revision": "search-feature-notebook-v1",
    "max_operations": int(os.environ.get("MAX_OPERATIONS", "500")),
    "fallback_distribution": os.environ.get("FALLBACK_DISTRIBUTION", "packaging"),
    "fallback_max_python_files": int(os.environ.get("FALLBACK_MAX_FILES", "40")),
    "max_doc_labels_per_operation": 8,
    "max_graph_neighbors_per_operation": 8,
    "enable_minhash": True,
    "minhash_num_perm": 64,
    "minhash_bands": 8,
    "simhash_bits": 64,
    "simhash_bands": 4,
    "lexical_top_k": 30,
    "dense_top_k": 30,
    "graph_seed_k": 8,
    "graph_top_k": 30,
    "final_top_k": 20,
    "rrf_rank_constant": 60,
    "embedding_planes": [
        {
            "plane_id": "identity",
            "fields": ["package_name", "module", "qualified_name", "kind"],
            "model_key": "hashing-char-512-v1",
        },
        {
            "plane_id": "documentation",
            "fields": ["qualified_name", "signature", "docstring"],
            "model_key": DOCUMENTATION_MODEL,
        },
        {
            "plane_id": "signature",
            "fields": ["qualified_name", "signature"],
            "model_key": "hashing-word-384-v1",
        },
    ],
    "queries": [
        "parse and compare Python package versions",
        "validate a dependency requirement and its environment marker",
        "canonicalize a Python project name",
    ],
    "random_seed": 17,
}

if CONFIG["max_operations"] <= 0:
    raise ValueError("max_operations must be positive")
if CONFIG["minhash_num_perm"] % CONFIG["minhash_bands"] != 0:
    raise ValueError("minhash_num_perm must be divisible by minhash_bands")
if CONFIG["simhash_bits"] % CONFIG["simhash_bands"] != 0:
    raise ValueError("simhash_bits must be divisible by simhash_bands")
for plane in CONFIG["embedding_planes"]:
    if plane["model_key"] not in MODEL_REGISTRY:
        raise KeyError(f"unknown model key: {plane['model_key']}")

CONFIG_DIGEST = digest_json(CONFIG)
RUN_STARTED = datetime.now(timezone.utc)
RUN_ID = RUN_STARTED.strftime("%Y%m%dT%H%M%S%fZ") + "-" + CONFIG_DIGEST.split(":")[-1][:10]
RUN_DIR = OUTPUT_ROOT / "search_feature_experiments" / RUN_ID
REGISTRY_DIR = OUTPUT_ROOT / "search_feature_registry"
RUN_DIR.mkdir(parents=True, exist_ok=False)
REGISTRY_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(CONFIG["random_seed"])
print(
    canonical_json(
        {
            "is_kaggle": IS_KAGGLE,
            "output_root": str(OUTPUT_ROOT.resolve()),
            "run_id": RUN_ID,
            "config_digest": CONFIG_DIGEST,
            "semantic_model_enabled": USE_SEMANTIC_MODEL,
        }
    )
)


## Optional repository bootstrap

Set REUSE_GITHUB_REPO to https://github.com/Amarel-Taylor-Scott/this-already-exists-dont-rebuild-it.git to clone this repository's source helpers. The URL is also shown as the documented default below, but cloning remains disabled unless REUSE_GITHUB_REPO is explicitly set. Bootstrap performs no editable installation and is not needed by the standalone notebook.


In [ ]:
DEFAULT_REPOSITORY_URL = (
    "https://github.com/Amarel-Taylor-Scott/"
    "this-already-exists-dont-rebuild-it.git"
)
REQUESTED_REPOSITORY_URL = os.environ.get("REUSE_GITHUB_REPO", "").strip()
BOOTSTRAP_CONFIG = {
    "enabled": bool(REQUESTED_REPOSITORY_URL),
    "repo_url": REQUESTED_REPOSITORY_URL or DEFAULT_REPOSITORY_URL,
    "git_ref": os.environ.get("REUSE_GITHUB_REF", "main"),
    "target": str(OUTPUT_ROOT / "existing-code-reuse-source"),
    "add_src_to_path": os.environ.get("REUSE_ADD_CLONED_SRC", "0") == "1",
}

if BOOTSTRAP_CONFIG["enabled"]:
    bootstrap_target = Path(BOOTSTRAP_CONFIG["target"])
    if not bootstrap_target.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                BOOTSTRAP_CONFIG["git_ref"],
                BOOTSTRAP_CONFIG["repo_url"],
                str(bootstrap_target),
            ],
            check=True,
        )
    elif not (bootstrap_target / ".git").is_dir():
        raise RuntimeError(f"bootstrap target exists but is not a Git checkout: {bootstrap_target}")
    if BOOTSTRAP_CONFIG["add_src_to_path"]:
        cloned_src = bootstrap_target / "src"
        if not cloned_src.is_dir():
            raise FileNotFoundError(f"cloned repository has no src directory: {cloned_src}")
        sys.path.insert(0, str(cloned_src))
    print(f"Trusted repository available at {bootstrap_target}")
else:
    print("Repository bootstrap disabled; using the standalone notebook implementation.")


## Locate or safely build the operation catalog

Catalog discovery prefers notebook 01 output at /kaggle/working/capability_catalog/catalog.sqlite, then Parquet mirrors and attached Kaggle inputs. The fallback reads installed distribution files and parses Python ASTs. It does not import the target distribution.


In [ ]:
OPERATION_COLUMNS = [
    "operation_id",
    "package_id",
    "package_name",
    "package_version",
    "module",
    "qualified_name",
    "kind",
    "signature",
    "docstring",
    "relative_path",
    "line_start",
    "line_end",
    "source_digest",
    "visibility",
    "extraction_method",
    "evidence_level",
]


def _existing_candidate(path: Path) -> Path | None:
    if path.is_file():
        return path
    if path.is_dir():
        preferred = path / "catalog.sqlite"
        if preferred.is_file():
            return preferred
        operations_parquet = path / "operations.parquet"
        if operations_parquet.is_file():
            return operations_parquet
    return None


def discover_catalog() -> Path | None:
    ordered_candidates: list[Path] = []
    if CATALOG_HINT:
        ordered_candidates.append(Path(CATALOG_HINT))
    ordered_candidates.extend(
        [
            Path("/kaggle/working/capability_catalog/catalog.sqlite"),
            Path("/kaggle/working/capability_catalog/operations.parquet"),
            Path("./capability_catalog/catalog.sqlite"),
            Path("./capability_catalog/operations.parquet"),
            Path("./.notebook_working/capability_catalog/catalog.sqlite"),
            Path("./.notebook_working/capability_catalog/operations.parquet"),
        ]
    )
    for candidate in ordered_candidates:
        found = _existing_candidate(candidate)
        if found is not None:
            return found

    if Path("/kaggle/input").is_dir():
        sqlite_candidates = sorted(Path("/kaggle/input").rglob("catalog.sqlite"))
        if sqlite_candidates:
            return sqlite_candidates[0]
        parquet_candidates = sorted(Path("/kaggle/input").rglob("operations.parquet"))
        if parquet_candidates:
            return parquet_candidates[0]
    return None


def _table_exists(connection: sqlite3.Connection, table_name: str) -> bool:
    row = connection.execute(
        "SELECT 1 FROM sqlite_master WHERE type IN ('table', 'view') AND name = ?",
        (table_name,),
    ).fetchone()
    return row is not None


def _read_sqlite_catalog(path: Path, limit: int) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    with sqlite3.connect(path) as connection:
        if not _table_exists(connection, "operations"):
            raise ValueError(f"SQLite file has no operations table: {path}")
        operations = pd.read_sql_query(
            "SELECT * FROM operations ORDER BY operation_id LIMIT ?",
            connection,
            params=(limit,),
        )
        selected_ids = operations["operation_id"].astype(str).tolist()
        if not selected_ids:
            return operations, pd.DataFrame(), pd.DataFrame()
        placeholders = ",".join("?" for _ in selected_ids)

        representations = (
            pd.read_sql_query(
                "SELECT * FROM representations "
                f"WHERE operation_id IN ({placeholders}) ORDER BY representation_id",
                connection,
                params=selected_ids,
            )
            if _table_exists(connection, "representations")
            else pd.DataFrame()
        )
        signals = (
            pd.read_sql_query(
                "SELECT * FROM signals "
                f"WHERE operation_id IN ({placeholders}) ORDER BY signal_id",
                connection,
                params=selected_ids,
            )
            if _table_exists(connection, "signals")
            else pd.DataFrame()
        )
    return operations, representations, signals


def _read_parquet_catalog(path: Path, limit: int) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    operations = pd.read_parquet(path).sort_values("operation_id").head(limit).reset_index(drop=True)
    selected_ids = set(operations["operation_id"].astype(str))
    parent = path.parent

    representation_path = parent / "representations.parquet"
    representations = pd.read_parquet(representation_path) if representation_path.is_file() else pd.DataFrame()
    if not representations.empty:
        representations = representations[
            representations["operation_id"].astype(str).isin(selected_ids)
        ].sort_values("representation_id").reset_index(drop=True)

    signal_path = parent / "signals.parquet"
    signals = pd.read_parquet(signal_path) if signal_path.is_file() else pd.DataFrame()
    if not signals.empty:
        signals = signals[
            signals["operation_id"].astype(str).isin(selected_ids)
        ].sort_values("signal_id").reset_index(drop=True)
    return operations, representations, signals


def _module_from_relative_path(relative_path: str) -> str:
    normalized = relative_path.replace("\\", "/")
    if normalized.endswith(".py"):
        normalized = normalized[:-3]
    parts = [part for part in normalized.split("/") if part and part != "__init__"]
    return ".".join(parts)


def _format_signature(node: ast.FunctionDef | ast.AsyncFunctionDef, qualified_name: str) -> str:
    arguments = ast.unparse(node.args)
    return_annotation = (
        " -> " + ast.unparse(node.returns) if node.returns is not None else ""
    )
    return f"{qualified_name}({arguments}){return_annotation}"


def _fallback_ast_catalog(
    distribution_name: str,
    max_files: int,
    max_operations: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    distribution = importlib.metadata.distribution(distribution_name)
    package_version = distribution.version
    package_id = f"pypi:{distribution.metadata['Name']}@{package_version}"
    records: list[dict[str, Any]] = []

    python_files = sorted(
        (
            file
            for file in (distribution.files or ())
            if str(file).endswith(".py") and ".dist-info/" not in str(file)
        ),
        key=str,
    )[:max_files]

    for distribution_file in python_files:
        source_path = Path(distribution.locate_file(distribution_file))
        try:
            source_bytes = source_path.read_bytes()
            source = source_bytes.decode("utf-8")
            tree = ast.parse(source, filename=str(distribution_file), type_comments=True)
        except (OSError, UnicodeDecodeError, SyntaxError):
            continue

        relative_path = str(distribution_file).replace("\\", "/")
        module = _module_from_relative_path(relative_path)
        source_digest = "sha256:" + hashlib.sha256(source_bytes).hexdigest()

        for node in tree.body:
            candidates: list[tuple[ast.AST, str, str]] = []
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and not node.name.startswith("_"):
                kind = "async_function" if isinstance(node, ast.AsyncFunctionDef) else "function"
                candidates.append((node, node.name, kind))
            elif isinstance(node, ast.ClassDef) and not node.name.startswith("_"):
                candidates.append((node, node.name, "class"))
                for child in node.body:
                    if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and not child.name.startswith("_"):
                        kind = "async_method" if isinstance(child, ast.AsyncFunctionDef) else "method"
                        candidates.append((child, f"{node.name}.{child.name}", kind))

            for candidate, qualified_name, kind in candidates:
                if isinstance(candidate, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    signature = _format_signature(candidate, qualified_name)
                else:
                    signature = f"{qualified_name}()"
                identity = {
                    "package_id": package_id,
                    "module": module,
                    "qualified_name": qualified_name,
                    "source_digest": source_digest,
                    "line_start": int(getattr(candidate, "lineno", 0)),
                }
                operation_id = (
                    f"{package_id}:{module}:{qualified_name}:"
                    + digest_json(identity).split(":")[-1][:16]
                )
                records.append(
                    {
                        "operation_id": operation_id,
                        "package_id": package_id,
                        "package_name": distribution.metadata["Name"],
                        "package_version": package_version,
                        "module": module,
                        "qualified_name": qualified_name,
                        "kind": kind,
                        "signature": signature,
                        "docstring": ast.get_docstring(candidate, clean=True) or "",
                        "relative_path": relative_path,
                        "line_start": int(getattr(candidate, "lineno", 0)),
                        "line_end": int(
                            getattr(candidate, "end_lineno", getattr(candidate, "lineno", 0))
                        ),
                        "source_digest": source_digest,
                        "visibility": "public",
                        "extraction_method": "notebook_ast_fallback_v1",
                        "evidence_level": "static_source",
                    }
                )
                if len(records) >= max_operations:
                    break
            if len(records) >= max_operations:
                break
        if len(records) >= max_operations:
            break

    if not records:
        raise RuntimeError(
            f"AST fallback found no public operations in installed distribution {distribution_name!r}"
        )
    operations = pd.DataFrame(records, columns=OPERATION_COLUMNS)
    operations = operations.sort_values("operation_id").reset_index(drop=True)
    return operations, pd.DataFrame(), pd.DataFrame()


def standardize_operations(frame: pd.DataFrame) -> pd.DataFrame:
    missing = sorted(set(OPERATION_COLUMNS) - set(frame.columns))
    if missing:
        raise ValueError(f"catalog operations are missing columns: {missing}")
    result = frame[OPERATION_COLUMNS].copy()
    for column in OPERATION_COLUMNS:
        if column in {"line_start", "line_end"}:
            result[column] = pd.to_numeric(result[column], errors="coerce").fillna(0).astype(int)
        else:
            result[column] = result[column].fillna("").astype(str)
    result = result.drop_duplicates("operation_id", keep="first")
    return result.sort_values("operation_id").head(CONFIG["max_operations"]).reset_index(drop=True)


catalog_started = time.perf_counter()
catalog_path = discover_catalog()
if catalog_path is None:
    operations, representations, signals = _fallback_ast_catalog(
        CONFIG["fallback_distribution"],
        CONFIG["fallback_max_python_files"],
        CONFIG["max_operations"],
    )
    catalog_source = {
        "kind": "ast_fallback",
        "distribution": CONFIG["fallback_distribution"],
        "safety": "static file reads and ast parsing only; no target-package import",
    }
else:
    if catalog_path.suffix.lower() in {".sqlite", ".db"}:
        operations, representations, signals = _read_sqlite_catalog(
            catalog_path,
            CONFIG["max_operations"],
        )
        source_kind = "sqlite"
    elif catalog_path.suffix.lower() == ".parquet":
        operations, representations, signals = _read_parquet_catalog(
            catalog_path,
            CONFIG["max_operations"],
        )
        source_kind = "parquet"
    else:
        raise ValueError(f"unsupported catalog artifact: {catalog_path}")
    catalog_source = {
        "kind": source_kind,
        "path": str(catalog_path.resolve()),
        "sha256": sha256_file(catalog_path),
    }

operations = standardize_operations(operations)
selected_operation_ids = set(operations["operation_id"])
if not representations.empty:
    representations = representations[
        representations["operation_id"].astype(str).isin(selected_operation_ids)
    ].reset_index(drop=True)
if not signals.empty:
    signals = signals[
        signals["operation_id"].astype(str).isin(selected_operation_ids)
    ].reset_index(drop=True)

if operations.empty:
    raise RuntimeError("catalog contains no operations after applying the scale limit")

print(
    canonical_json(
        {
            "catalog_source": catalog_source,
            "operations": len(operations),
            "representations": len(representations),
            "signals": len(signals),
            "load_seconds": round(time.perf_counter() - catalog_started, 6),
        }
    )
)
operations.head(5)


## Append-only feature registry

The registry is a partitioned long table. A run writes a new immutable Parquet partition containing only previously unseen feature IDs. Existing partitions are read for deduplication and are never rewritten. Flexible values and evidence are canonical JSON rather than fixed feature-specific columns.


In [ ]:
FEATURE_COLUMNS = [
    "feature_id",
    "entity_kind",
    "entity_id",
    "feature_kind",
    "namespace",
    "value_type",
    "value_text",
    "value_json",
    "generator",
    "generator_revision",
    "model_id",
    "model_revision",
    "source_ref_json",
    "evidence_json",
    "generator_config_json",
    "generator_config_digest",
    "experiment_config_digest",
    "artifact_uri",
    "artifact_row",
    "confidence",
    "review_status",
    "created_at_utc",
    "run_id",
]


class AppendOnlyFeatureRegistry:
    def __init__(
        self,
        partition_dir: Path,
        run_dir: Path,
        run_id: str,
        experiment_config_digest: str,
    ) -> None:
        self.partition_dir = partition_dir
        self.run_dir = run_dir
        self.run_id = run_id
        self.experiment_config_digest = experiment_config_digest
        self.created_at = RUN_STARTED.isoformat()
        self.new_rows: list[dict[str, Any]] = []
        self.existing_ids: set[str] = set()

        for partition in sorted(partition_dir.glob("part-*.parquet")):
            prior_ids = pd.read_parquet(partition, columns=["feature_id"])
            self.existing_ids.update(prior_ids["feature_id"].astype(str))

    def add(
        self,
        *,
        entity_kind: str,
        entity_id: str,
        feature_kind: str,
        namespace: str,
        value: Any,
        value_type: str,
        generator: str,
        generator_revision: str,
        evidence: Sequence[Mapping[str, Any]],
        generator_config: Mapping[str, Any],
        source_refs: Sequence[Mapping[str, Any]] | None = None,
        model_id: str = "",
        model_revision: str = "",
        artifact_uri: str = "",
        artifact_row: int = -1,
        confidence: float = 1.0,
        review_status: str = "derived",
    ) -> str:
        if not entity_kind or not entity_id or not feature_kind or not namespace:
            raise ValueError("feature identity fields cannot be empty")
        if not generator or not generator_revision:
            raise ValueError("generator provenance cannot be empty")
        if not evidence:
            raise ValueError("every feature requires at least one evidence reference")
        if model_id and not model_revision:
            raise ValueError("model-backed features require a model revision")
        if not 0.0 <= float(confidence) <= 1.0:
            raise ValueError("feature confidence must be in [0, 1]")
        if not review_status:
            raise ValueError("review_status cannot be empty")

        value_json = canonical_json(value)
        source_ref_json = canonical_json(list(source_refs or evidence))
        evidence_json = canonical_json(list(evidence))
        generator_config_json = canonical_json(dict(generator_config))
        generator_config_digest = digest_json(dict(generator_config))
        identity = {
            "entity_kind": entity_kind,
            "entity_id": entity_id,
            "feature_kind": feature_kind,
            "namespace": namespace,
            "value_type": value_type,
            "value_json": value_json,
            "generator": generator,
            "generator_revision": generator_revision,
            "model_id": model_id,
            "model_revision": model_revision,
            "source_ref_json": source_ref_json,
            "evidence_json": evidence_json,
            "confidence": float(confidence),
            "review_status": review_status,
            "generator_config_digest": generator_config_digest,
        }
        feature_id = digest_json(identity)

        if feature_id in self.existing_ids:
            return feature_id
        if any(row["feature_id"] == feature_id for row in self.new_rows):
            return feature_id

        value_text = value if isinstance(value, str) else ""
        self.new_rows.append(
            {
                "feature_id": feature_id,
                "entity_kind": entity_kind,
                "entity_id": entity_id,
                "feature_kind": feature_kind,
                "namespace": namespace,
                "value_type": value_type,
                "value_text": value_text,
                "value_json": value_json,
                "generator": generator,
                "generator_revision": generator_revision,
                "model_id": model_id,
                "model_revision": model_revision,
                "source_ref_json": source_ref_json,
                "evidence_json": evidence_json,
                "generator_config_json": generator_config_json,
                "generator_config_digest": generator_config_digest,
                "experiment_config_digest": self.experiment_config_digest,
                "artifact_uri": artifact_uri,
                "artifact_row": int(artifact_row),
                "confidence": float(confidence),
                "review_status": review_status,
                "created_at_utc": self.created_at,
                "run_id": self.run_id,
            }
        )
        return feature_id

    def flush(self) -> tuple[Path, Path, pd.DataFrame]:
        append_frame = pd.DataFrame(self.new_rows, columns=FEATURE_COLUMNS)
        append_path = self.run_dir / "feature_append.parquet"
        append_frame.to_parquet(append_path, index=False)

        if not append_frame.empty:
            partition_path = self.partition_dir / f"part-{self.run_id}.parquet"
            if partition_path.exists():
                raise FileExistsError(partition_path)
            append_frame.to_parquet(partition_path, index=False)

        partitions = sorted(self.partition_dir.glob("part-*.parquet"))
        frames = [pd.read_parquet(partition) for partition in partitions]
        snapshot = (
            pd.concat(frames, ignore_index=True)
            if frames
            else pd.DataFrame(columns=FEATURE_COLUMNS)
        )
        if not snapshot.empty and snapshot["feature_id"].duplicated().any():
            duplicates = snapshot.loc[snapshot["feature_id"].duplicated(), "feature_id"].tolist()
            raise AssertionError(f"append-only registry contains duplicate IDs: {duplicates[:5]}")
        snapshot_path = self.run_dir / "feature_registry_snapshot.parquet"
        snapshot.to_parquet(snapshot_path, index=False)
        return append_path, snapshot_path, snapshot


registry = AppendOnlyFeatureRegistry(
    REGISTRY_DIR,
    RUN_DIR,
    RUN_ID,
    CONFIG_DIGEST,
)
print(f"Registry already contains {len(registry.existing_ids):,} feature IDs")


## Deterministic labels, keys, phrases, ports, and locality-sensitive blocks

All normalization rules are versioned. Semantic-looking phrases are explicitly marked as deterministic templates; they are not human judgments. Type-flow edges are candidates based on matching explicit annotations, not verified compatibility claims.


In [ ]:
NORMALIZER_REVISION = "unicode-nfkc-casefold-token-v1"
LABEL_GENERATOR_REVISION = "identity-doc-keywords-v1"
PORT_GENERATOR_REVISION = "python-signature-ast-v1"
SIMHASH_REVISION = "blake2b-token-simhash-64-v1"
MINHASH_REVISION = "datasketch-token-bigram-v1"

STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "if", "in",
    "into", "is", "it", "of", "on", "or", "that", "the", "this", "to", "with",
    "return", "returns", "given", "using", "use",
}


def normalize_text(value: str) -> str:
    normalized = unicodedata.normalize("NFKC", str(value)).casefold()
    return " ".join(re.findall(r"[a-z0-9]+", normalized))


def identifier_tokens(value: str) -> list[str]:
    split_camel = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", str(value))
    return normalize_text(split_camel.replace("_", " ").replace(".", " ")).split()


def first_sentence(value: str) -> str:
    compact = " ".join(str(value).split())
    if not compact:
        return ""
    pieces = re.split(r"(?<=[.!?])\s+", compact, maxsplit=1)
    return pieces[0][:500]


def documentation_keywords(value: str, limit: int) -> list[str]:
    tokens = [
        token
        for token in normalize_text(value).split()
        if len(token) >= 3 and token not in STOPWORDS and not token.isdigit()
    ]
    counts = Counter(tokens)
    return [
        token
        for token, _ in sorted(counts.items(), key=lambda item: (-item[1], item[0]))[:limit]
    ]


def parse_signature_ports(signature: str) -> tuple[list[dict[str, str]], dict[str, str] | None]:
    start = str(signature).find("(")
    if start < 0:
        return [], None
    signature_tail = str(signature)[start:]
    try:
        parsed = ast.parse(f"def _operation{signature_tail}:\n    pass\n")
        function = parsed.body[0]
        if not isinstance(function, ast.FunctionDef):
            return [], None
    except SyntaxError:
        return [], None

    inputs: list[dict[str, str]] = []
    position = 0
    positional = [*function.args.posonlyargs, *function.args.args]
    for argument in positional:
        inputs.append(
            {
                "name": argument.arg,
                "annotation": ast.unparse(argument.annotation) if argument.annotation else "",
                "parameter_kind": "positional",
                "position": str(position),
            }
        )
        position += 1
    if function.args.vararg:
        argument = function.args.vararg
        inputs.append(
            {
                "name": argument.arg,
                "annotation": ast.unparse(argument.annotation) if argument.annotation else "",
                "parameter_kind": "var_positional",
                "position": str(position),
            }
        )
        position += 1
    for argument in function.args.kwonlyargs:
        inputs.append(
            {
                "name": argument.arg,
                "annotation": ast.unparse(argument.annotation) if argument.annotation else "",
                "parameter_kind": "keyword_only",
                "position": str(position),
            }
        )
        position += 1
    if function.args.kwarg:
        argument = function.args.kwarg
        inputs.append(
            {
                "name": argument.arg,
                "annotation": ast.unparse(argument.annotation) if argument.annotation else "",
                "parameter_kind": "var_keyword",
                "position": str(position),
            }
        )

    output = (
        {"annotation": ast.unparse(function.returns)}
        if function.returns is not None
        else None
    )
    return inputs, output


def simhash64(tokens: Sequence[str]) -> int:
    vector = [0] * 64
    for token in sorted(tokens):
        token_hash = int.from_bytes(
            hashlib.blake2b(token.encode("utf-8"), digest_size=8).digest(),
            "big",
        )
        for bit in range(64):
            vector[bit] += 1 if token_hash & (1 << bit) else -1
    fingerprint = 0
    for bit, weight in enumerate(vector):
        if weight >= 0:
            fingerprint |= 1 << bit
    return fingerprint


def simhash_block_keys(tokens: Sequence[str], bands: int) -> list[str]:
    fingerprint = simhash64(tokens)
    width = 64 // bands
    mask = (1 << width) - 1
    return [
        f"simhash:b{band}:{((fingerprint >> (band * width)) & mask):0{width // 4}x}"
        for band in range(bands)
    ]


def minhash_block_keys(tokens: Sequence[str], num_perm: int, bands: int) -> list[str]:
    shingles = set(tokens)
    shingles.update(
        f"{left}::{right}" for left, right in zip(tokens, tokens[1:])
    )
    if not shingles:
        shingles = {"empty"}
    sketch = MinHash(num_perm=num_perm, seed=CONFIG["random_seed"])
    for shingle in sorted(shingles):
        sketch.update(shingle.encode("utf-8"))
    rows_per_band = num_perm // bands
    keys: list[str] = []
    for band in range(bands):
        start = band * rows_per_band
        end = start + rows_per_band
        band_bytes = np.asarray(sketch.hashvalues[start:end], dtype=">u8").tobytes()
        keys.append(
            f"minhash:b{band}:"
            + hashlib.sha256(band_bytes).hexdigest()[:20]
        )
    return keys


def evidence_for(operation: Mapping[str, Any], fields: Sequence[str]) -> list[dict[str, Any]]:
    return [
        {
            "record_type": "operation",
            "operation_id": str(operation["operation_id"]),
            "source_digest": str(operation["source_digest"]),
            "fields": list(fields),
            "extraction_method": str(operation["extraction_method"]),
            "evidence_level": str(operation["evidence_level"]),
        }
    ]


def json_list(value: Any) -> list[str]:
    if isinstance(value, (list, tuple, np.ndarray)):
        return [str(item) for item in value]
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    text = str(value).strip()
    if not text:
        return []
    try:
        decoded = json.loads(text)
    except json.JSONDecodeError:
        return [text]
    return [str(item) for item in decoded] if isinstance(decoded, list) else [str(decoded)]


operation_rows = operations.to_dict(orient="records")
operation_by_id = {row["operation_id"]: row for row in operation_rows}
labels_by_operation: dict[str, list[str]] = {}
phrases_by_operation: dict[str, list[str]] = {}
ports_by_operation: dict[str, dict[str, Any]] = {}
blocking_keys_by_operation: dict[str, list[str]] = {}

feature_generation_started = time.perf_counter()

# Preserve notebook 01 representations and signals as seed feature rows.
for row in representations.to_dict(orient="records"):
    operation_id = str(row["operation_id"])
    source_fields = json_list(row.get("source_fields", row.get("source_fields_json", [])))
    registry.add(
        entity_kind="operation",
        entity_id=operation_id,
        feature_kind="catalog_representation",
        namespace=str(row.get("plane", "unknown")),
        value=str(row.get("text", "")),
        value_type="text",
        generator=str(row.get("generator", "catalog")),
        generator_revision=str(row.get("generator_version", "unknown")),
        evidence=[
            {
                "record_type": "representation",
                "representation_id": str(row.get("representation_id", "")),
                "source_digest": str(row.get("source_digest", "")),
                "source_fields": source_fields,
                "review_status": str(row.get("review_status", "")),
            }
        ],
        generator_config={"source": "notebook_01_catalog"},
        confidence=1.0,
        review_status=str(row.get("review_status", "observed")) or "observed",
    )

for row in signals.to_dict(orient="records"):
    operation_id = str(row["operation_id"])
    evidence_fields = json_list(row.get("evidence_fields", row.get("evidence_fields_json", [])))
    registry.add(
        entity_kind="operation",
        entity_id=operation_id,
        feature_kind=str(row.get("signal_kind", "catalog_signal")),
        namespace=str(row.get("namespace", "unknown")),
        value=str(row.get("value", "")),
        value_type="text",
        generator=str(row.get("generator", "catalog")),
        generator_revision=str(row.get("generator_version", "unknown")),
        evidence=[
            {
                "record_type": "signal",
                "signal_id": str(row.get("signal_id", "")),
                "operation_id": operation_id,
                "confidence": float(row.get("confidence", 0.0)),
                "evidence_fields": evidence_fields,
            }
        ],
        generator_config={"source": "notebook_01_catalog"},
        confidence=float(row.get("confidence", 0.0)),
        review_status="derived",
    )

for operation in operation_rows:
    operation_id = operation["operation_id"]
    base_evidence = evidence_for(operation, ["package_name", "module", "qualified_name"])

    for field_name in [
        "package_name",
        "package_version",
        "module",
        "qualified_name",
        "kind",
        "visibility",
        "signature",
        "evidence_level",
    ]:
        registry.add(
            entity_kind="operation",
            entity_id=operation_id,
            feature_kind="attribute",
            namespace=field_name,
            value=operation[field_name],
            value_type="text",
            generator="catalog_projection",
            generator_revision="operation-attribute-v1",
            evidence=evidence_for(operation, [field_name]),
            generator_config={"normalization": "none"},
            confidence=1.0,
            review_status="observed",
        )

    identity_labels = sorted(
        set(
            identifier_tokens(operation["qualified_name"])
            + identifier_tokens(operation["module"])
            + identifier_tokens(operation["package_name"])
        )
    )
    doc_labels = documentation_keywords(
        operation["docstring"],
        CONFIG["max_doc_labels_per_operation"],
    )
    labels = sorted(set(identity_labels + doc_labels))
    labels_by_operation[operation_id] = labels

    for label in labels:
        registry.add(
            entity_kind="operation",
            entity_id=operation_id,
            feature_kind="normalized_label",
            namespace="identity_or_documentation_token",
            value=label,
            value_type="text",
            generator="deterministic_text_normalizer",
            generator_revision=LABEL_GENERATOR_REVISION,
            evidence=evidence_for(
                operation,
                ["qualified_name", "module", "package_name", "docstring"],
            ),
            generator_config={
                "normalizer_revision": NORMALIZER_REVISION,
                "stopwords_digest": digest_json(sorted(STOPWORDS)),
                "max_doc_labels": CONFIG["max_doc_labels_per_operation"],
            },
        )

    tags = [
        f"package:{normalize_text(operation['package_name']).replace(' ', '-')}",
        f"kind:{normalize_text(operation['kind']).replace(' ', '-')}",
        f"visibility:{normalize_text(operation['visibility'])}",
        f"evidence:{normalize_text(operation['evidence_level']).replace(' ', '-')}",
    ]
    for tag in tags:
        registry.add(
            entity_kind="operation",
            entity_id=operation_id,
            feature_kind="tag",
            namespace=tag.split(":", 1)[0],
            value=tag,
            value_type="text",
            generator="deterministic_catalog_tags",
            generator_revision="catalog-tags-v1",
            evidence=base_evidence,
            generator_config={"normalizer_revision": NORMALIZER_REVISION},
        )

    symbol_words = " ".join(identifier_tokens(operation["qualified_name"]))
    sentence = first_sentence(operation["docstring"])
    search_phrases = sorted(
        {
            phrase
            for phrase in [
                operation["qualified_name"],
                f"{operation['module']}.{operation['qualified_name']}",
                f"{operation['package_name']} {symbol_words}",
                symbol_words,
                sentence,
            ]
            if phrase
        }
    )
    semantic_phrase = (
        f"Use {operation['qualified_name']} from {operation['package_name']}"
        + (f" to {sentence[0].lower() + sentence[1:]}" if sentence else "")
    )
    phrases_by_operation[operation_id] = [*search_phrases, semantic_phrase]

    for phrase in search_phrases:
        registry.add(
            entity_kind="operation",
            entity_id=operation_id,
            feature_kind="search_phrase",
            namespace="identity_or_observed_documentation",
            value=phrase,
            value_type="text",
            generator="deterministic_phrase_projection",
            generator_revision="search-phrase-v1",
            evidence=evidence_for(
                operation,
                ["package_name", "module", "qualified_name", "docstring"],
            ),
            generator_config={"first_sentence_limit": 500},
        )
    registry.add(
        entity_kind="operation",
        entity_id=operation_id,
        feature_kind="semantic_phrase",
        namespace="purpose_template",
        value=semantic_phrase,
        value_type="text",
        generator="deterministic_semantic_template",
        generator_revision="package-symbol-purpose-v1",
        evidence=evidence_for(
            operation,
            ["package_name", "qualified_name", "docstring"],
        ),
        generator_config={"template": "Use SYMBOL from PACKAGE to FIRST_SENTENCE"},
    )

    inputs, output = parse_signature_ports(operation["signature"])
    ports_by_operation[operation_id] = {"inputs": inputs, "output": output}
    for input_port in inputs:
        registry.add(
            entity_kind="operation",
            entity_id=operation_id,
            feature_kind="input_port",
            namespace=input_port["name"],
            value=input_port,
            value_type="json",
            generator="python_signature_parser",
            generator_revision=PORT_GENERATOR_REVISION,
            evidence=evidence_for(operation, ["signature"]),
            generator_config={"parser": "python_ast"},
        )
    if output is not None:
        registry.add(
            entity_kind="operation",
            entity_id=operation_id,
            feature_kind="output_port",
            namespace="return_annotation",
            value=output,
            value_type="json",
            generator="python_signature_parser",
            generator_revision=PORT_GENERATOR_REVISION,
            evidence=evidence_for(operation, ["signature"]),
            generator_config={"parser": "python_ast"},
        )

    onion_layers = {
        "0_identity": " ".join(
            [operation["package_name"], operation["module"], operation["qualified_name"]]
        ),
        "1_contract": " ".join(
            value
            for value in [operation["qualified_name"], operation["signature"], sentence]
            if value
        ),
        "2_documentation": operation["docstring"],
        "3_catalog_context": " ".join(
            [
                operation["package_name"],
                operation["package_version"],
                operation["module"],
                operation["kind"],
                operation["evidence_level"],
            ]
        ),
    }
    for layer_name, layer_text in onion_layers.items():
        if not layer_text:
            continue
        registry.add(
            entity_kind="operation",
            entity_id=operation_id,
            feature_kind="onion_layer",
            namespace=layer_name,
            value=layer_text,
            value_type="text",
            generator="deterministic_onion_projection",
            generator_revision="operation-onion-v1",
            evidence=evidence_for(
                operation,
                [
                    "package_name",
                    "package_version",
                    "module",
                    "qualified_name",
                    "signature",
                    "docstring",
                    "kind",
                    "evidence_level",
                ],
            ),
            generator_config={"layers": list(onion_layers)},
        )

    blocking_keys = {
        f"package:{normalize_text(operation['package_name']).replace(' ', '-')}",
        f"module:{normalize_text(operation['module']).replace(' ', '.')}",
        f"kind:{normalize_text(operation['kind']).replace(' ', '-')}",
    }
    if identity_labels:
        blocking_keys.add(f"name-token:{identity_labels[0]}")
    for keyword in doc_labels[:3]:
        blocking_keys.add(f"doc-keyword:{keyword}")
    for input_port in inputs:
        if input_port["annotation"]:
            blocking_keys.add(f"input-type:{normalize_text(input_port['annotation'])}")
    if output and output["annotation"]:
        blocking_keys.add(f"output-type:{normalize_text(output['annotation'])}")

    lsh_tokens = labels or [normalize_text(operation_id)]
    blocking_keys.update(
        simhash_block_keys(lsh_tokens, CONFIG["simhash_bands"])
    )
    if CONFIG["enable_minhash"]:
        blocking_keys.update(
            minhash_block_keys(
                lsh_tokens,
                CONFIG["minhash_num_perm"],
                CONFIG["minhash_bands"],
            )
        )
    ordered_blocking_keys = sorted(blocking_keys)
    blocking_keys_by_operation[operation_id] = ordered_blocking_keys

    for blocking_key in ordered_blocking_keys:
        if blocking_key.startswith("simhash:"):
            generator = "simhash"
            revision = SIMHASH_REVISION
        elif blocking_key.startswith("minhash:"):
            generator = "minhash"
            revision = MINHASH_REVISION
        else:
            generator = "deterministic_blocking_key"
            revision = "catalog-and-port-blocks-v1"
        registry.add(
            entity_kind="operation",
            entity_id=operation_id,
            feature_kind="blocking_key",
            namespace=blocking_key.split(":", 1)[0],
            value=blocking_key,
            value_type="text",
            generator=generator,
            generator_revision=revision,
            evidence=evidence_for(
                operation,
                [
                    "package_name",
                    "module",
                    "kind",
                    "qualified_name",
                    "docstring",
                    "signature",
                ],
            ),
            generator_config={
                "normalizer_revision": NORMALIZER_REVISION,
                "simhash_bands": CONFIG["simhash_bands"],
                "minhash_num_perm": CONFIG["minhash_num_perm"],
                "minhash_bands": CONFIG["minhash_bands"],
            },
        )

print(
    canonical_json(
        {
            "new_feature_rows_before_edges_and_embeddings": len(registry.new_rows),
            "generation_seconds": round(time.perf_counter() - feature_generation_started, 6),
        }
    )
)


## Candidate graph

Graph edges are explicitly typed. Same-module edges are contextual and undirected. Annotation-flow edges are directional candidates derived only from exact normalized return/input annotations; they remain unverified until an execution-backed compatibility test exists.


In [ ]:
def normalized_annotation(value: str) -> str:
    tokens = [
        token
        for token in normalize_text(value).split()
        if token not in {"typing", "optional", "union", "none"}
    ]
    return " ".join(tokens)


edge_rows: list[dict[str, Any]] = []
edge_keys: set[tuple[str, str, str]] = set()


def record_edge(
    source: str,
    target: str,
    relation: str,
    *,
    directed: bool,
    evidence: Sequence[Mapping[str, Any]],
) -> None:
    if source == target:
        return
    canonical_source, canonical_target = source, target
    if not directed and canonical_target < canonical_source:
        canonical_source, canonical_target = canonical_target, canonical_source
    edge_key = (canonical_source, canonical_target, relation)
    if edge_key in edge_keys:
        return
    edge_keys.add(edge_key)
    edge_id = digest_json(
        {
            "source": canonical_source,
            "target": canonical_target,
            "relation": relation,
            "directed": directed,
        }
    )
    edge = {
        "edge_id": edge_id,
        "source_operation_id": canonical_source,
        "target_operation_id": canonical_target,
        "relation": relation,
        "directed": bool(directed),
        "verified": False,
        "generator": "deterministic_candidate_graph",
        "generator_revision": "module-and-annotation-edges-v1",
        "evidence_json": canonical_json(list(evidence)),
    }
    edge_rows.append(edge)
    registry.add(
        entity_kind="edge",
        entity_id=edge_id,
        feature_kind="edge",
        namespace=relation,
        value={
            "source_operation_id": canonical_source,
            "target_operation_id": canonical_target,
            "directed": bool(directed),
            "verified": False,
        },
        value_type="json",
        generator=edge["generator"],
        generator_revision=edge["generator_revision"],
        evidence=evidence,
        generator_config={
            "max_graph_neighbors_per_operation": CONFIG[
                "max_graph_neighbors_per_operation"
            ],
            "annotation_normalizer": NORMALIZER_REVISION,
        },
    )


operations_by_module: dict[str, list[str]] = defaultdict(list)
for operation in operation_rows:
    operations_by_module[operation["module"]].append(operation["operation_id"])

for module, operation_ids in sorted(operations_by_module.items()):
    ordered = sorted(operation_ids)
    for index, source in enumerate(ordered):
        neighbors = [
            target
            for target in ordered
            if target != source
        ][: CONFIG["max_graph_neighbors_per_operation"]]
        for target in neighbors:
            record_edge(
                source,
                target,
                "same_module_context",
                directed=False,
                evidence=[
                    {
                        "record_type": "operation_pair",
                        "source_operation_id": source,
                        "target_operation_id": target,
                        "field": "module",
                        "value": module,
                    }
                ],
            )

producers_by_annotation: dict[str, list[str]] = defaultdict(list)
consumers_by_annotation: dict[str, list[str]] = defaultdict(list)
for operation_id, ports in ports_by_operation.items():
    output = ports["output"]
    if output and output["annotation"]:
        annotation = normalized_annotation(output["annotation"])
        if annotation:
            producers_by_annotation[annotation].append(operation_id)
    for input_port in ports["inputs"]:
        if input_port["annotation"]:
            annotation = normalized_annotation(input_port["annotation"])
            if annotation:
                consumers_by_annotation[annotation].append(operation_id)

for annotation in sorted(set(producers_by_annotation) & set(consumers_by_annotation)):
    for source in sorted(producers_by_annotation[annotation]):
        targets = [
            target
            for target in sorted(consumers_by_annotation[annotation])
            if target != source
        ][: CONFIG["max_graph_neighbors_per_operation"]]
        for target in targets:
            record_edge(
                source,
                target,
                "annotation_flow_candidate",
                directed=True,
                evidence=[
                    {
                        "record_type": "signature_pair",
                        "source_operation_id": source,
                        "target_operation_id": target,
                        "normalized_annotation": annotation,
                        "verified": False,
                    }
                ],
            )

edges = pd.DataFrame(
    edge_rows,
    columns=[
        "edge_id",
        "source_operation_id",
        "target_operation_id",
        "relation",
        "directed",
        "verified",
        "generator",
        "generator_revision",
        "evidence_json",
    ],
)
print(
    canonical_json(
        {
            "edges": len(edges),
            "same_module_edges": int(
                (edges["relation"] == "same_module_context").sum()
            ) if not edges.empty else 0,
            "annotation_flow_candidates": int(
                (edges["relation"] == "annotation_flow_candidate").sum()
            ) if not edges.empty else 0,
            "verified_edges": int(edges["verified"].sum()) if not edges.empty else 0,
        }
    )
)


## Field-specific embedding registry

Each plane chooses a model through MODEL_REGISTRY. The default hashing encoders are deterministic, CPU-friendly, and require no model download. Set USE_SENTENCE_TRANSFORMER=1 to use the pinned MiniLM revision for the documentation plane. Every vector row is linked back to its artifact, model, revision, fields, and generator configuration.


In [ ]:
_MODEL_CACHE: dict[str, Any] = {}


def text_for_plane(operation: Mapping[str, Any], fields: Sequence[str]) -> str:
    return "\n".join(
        f"{field}: {operation.get(field, '')}"
        for field in fields
        if str(operation.get(field, "")).strip()
    )


def encode_texts(texts: Sequence[str], model_key: str) -> np.ndarray:
    spec = MODEL_REGISTRY[model_key]
    backend = spec["backend"]
    if backend == "sklearn_hashing":
        vectorizer = HashingVectorizer(
            n_features=int(spec["dimension"]),
            analyzer=spec["analyzer"],
            ngram_range=tuple(spec["ngram_range"]),
            alternate_sign=bool(spec["alternate_sign"]),
            norm="l2",
            lowercase=True,
        )
        return vectorizer.transform(list(texts)).toarray().astype(np.float32)

    if backend == "sentence_transformers":
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError as error:
            raise RuntimeError(
                "A sentence-transformer model was selected, but sentence-transformers "
                "is unavailable. Re-run the install cell."
            ) from error
        if model_key not in _MODEL_CACHE:
            _MODEL_CACHE[model_key] = SentenceTransformer(
                spec["model_id"],
                revision=spec["revision"],
            )
        matrix = _MODEL_CACHE[model_key].encode(
            list(texts),
            batch_size=int(spec["batch_size"]),
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=len(texts) >= 100,
        )
        matrix = np.asarray(matrix, dtype=np.float32)
        if matrix.shape[1] != int(spec["dimension"]):
            raise ValueError(
                f"model {model_key} emitted dimension {matrix.shape[1]}, "
                f"expected {spec['dimension']}"
            )
        return matrix

    raise ValueError(f"unsupported embedding backend: {backend}")


def safe_slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", value.casefold()).strip("-")


embedding_spaces: dict[str, dict[str, Any]] = {}
embedding_manifest: list[dict[str, Any]] = []
embedding_started = time.perf_counter()

for plane_config in CONFIG["embedding_planes"]:
    plane_id = plane_config["plane_id"]
    fields = plane_config["fields"]
    model_key = plane_config["model_key"]
    model_spec = MODEL_REGISTRY[model_key]
    texts = [text_for_plane(operation, fields) for operation in operation_rows]
    matrix = encode_texts(texts, model_key)
    if matrix.shape != (len(operation_rows), int(model_spec["dimension"])):
        raise AssertionError(
            f"unexpected embedding shape for {plane_id}: {matrix.shape}"
        )

    vector_path = RUN_DIR / (
        f"embeddings-{safe_slug(plane_id)}-{safe_slug(model_key)}.npy"
    )
    np.save(vector_path, matrix, allow_pickle=False)
    matrix_digest = sha256_file(vector_path)
    embedding_spaces[plane_id] = {
        "matrix": matrix,
        "texts": texts,
        "model_key": model_key,
        "fields": fields,
        "artifact_path": vector_path,
    }
    embedding_manifest.append(
        {
            "plane_id": plane_id,
            "fields": fields,
            "model_key": model_key,
            "model_spec": model_spec,
            "shape": list(matrix.shape),
            "dtype": str(matrix.dtype),
            "normalized": True,
            "artifact": str(vector_path),
            "sha256": matrix_digest,
        }
    )

    for row_index, operation in enumerate(operation_rows):
        registry.add(
            entity_kind="operation",
            entity_id=operation["operation_id"],
            feature_kind="embedding",
            namespace=plane_id,
            value={
                "dimension": int(matrix.shape[1]),
                "dtype": str(matrix.dtype),
                "normalized": True,
                "fields": fields,
            },
            value_type="vector_reference",
            generator="field_specific_embedding",
            generator_revision="embedding-plane-v1",
            model_id=model_spec["model_id"],
            model_revision=model_spec["revision"],
            evidence=evidence_for(operation, fields),
            generator_config={
                "plane_id": plane_id,
                "fields": fields,
                "model_key": model_key,
                "model_spec": model_spec,
            },
            artifact_uri=str(vector_path),
            artifact_row=row_index,
        )

embedding_manifest_path = RUN_DIR / "embedding_manifest.json"
embedding_manifest_path.write_text(
    json.dumps(embedding_manifest, indent=2, sort_keys=True),
    encoding="utf-8",
)
print(
    canonical_json(
        {
            "embedding_planes": len(embedding_manifest),
            "vectors": sum(item["shape"][0] for item in embedding_manifest),
            "embedding_seconds": round(time.perf_counter() - embedding_started, 6),
        }
    )
)


## Lexical, dense, and graph candidate generation

The candidate experiment uses word TF-IDF, character TF-IDF, all configured embedding planes, and one-hop graph expansion. Reciprocal-rank fusion keeps source rank, raw score, and reason provenance. There are no relevance labels in this notebook, so it emits candidates and measured run diagnostics but does not invent Recall, MRR, or acceptance metrics.


In [ ]:
operation_ids = [operation["operation_id"] for operation in operation_rows]
search_texts = [
    "\n".join(
        [
            operation["package_name"],
            operation["module"],
            operation["qualified_name"],
            operation["signature"],
            operation["docstring"],
            *phrases_by_operation[operation["operation_id"]],
            *labels_by_operation[operation["operation_id"]],
        ]
    )
    for operation in operation_rows
]

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    sublinear_tf=True,
    strip_accents="unicode",
)
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    sublinear_tf=True,
    strip_accents="unicode",
)
word_matrix = word_vectorizer.fit_transform(search_texts)
char_matrix = char_vectorizer.fit_transform(search_texts)


def stable_top_scores(
    scores: np.ndarray,
    limit: int,
    *,
    minimum_score: float = 0.0,
) -> list[tuple[str, float]]:
    ranked = [
        (operation_id, float(score))
        for operation_id, score in zip(operation_ids, scores, strict=True)
        if float(score) > minimum_score
    ]
    ranked.sort(key=lambda item: (-item[1], item[0]))
    return ranked[:limit]


graph = nx.DiGraph()
graph.add_nodes_from(operation_ids)
for edge in edge_rows:
    graph.add_edge(
        edge["source_operation_id"],
        edge["target_operation_id"],
        relation=edge["relation"],
        verified=edge["verified"],
    )
    if not edge["directed"]:
        graph.add_edge(
            edge["target_operation_id"],
            edge["source_operation_id"],
            relation=edge["relation"],
            verified=edge["verified"],
        )


def exact_candidates(query: str) -> list[tuple[str, float, str]]:
    normalized_query = normalize_text(query)
    matches: list[tuple[str, float, str]] = []
    for operation in operation_rows:
        operation_id = operation["operation_id"]
        aliases = {
            normalize_text(operation["operation_id"]),
            normalize_text(operation["qualified_name"]),
            normalize_text(
                f"{operation['module']}.{operation['qualified_name']}"
            ),
            *(
                normalize_text(phrase)
                for phrase in phrases_by_operation[operation_id]
                if phrase
            ),
        }
        best_alias = ""
        best_score = 0.0
        for alias in aliases:
            if not alias:
                continue
            if normalized_query == alias:
                score = 1.0
            elif f" {alias} " in f" {normalized_query} ":
                score = 0.9
            else:
                continue
            if score > best_score or (score == best_score and alias < best_alias):
                best_score = score
                best_alias = alias
        if best_score > 0:
            matches.append((operation_id, best_score, f"matched_alias:{best_alias}"))
    matches.sort(key=lambda item: (-item[1], item[0]))
    return matches[: CONFIG["lexical_top_k"]]


def graph_candidates(
    source_rankings: Mapping[str, Sequence[tuple[str, float, str]]],
) -> list[tuple[str, float, str]]:
    seed_scores: dict[str, float] = defaultdict(float)
    seed_reasons: dict[str, list[str]] = defaultdict(list)
    for source_name, ranking in sorted(source_rankings.items()):
        for rank, (operation_id, _raw_score, _reason) in enumerate(
            ranking[: CONFIG["graph_seed_k"]],
            start=1,
        ):
            contribution = 1.0 / (CONFIG["rrf_rank_constant"] + rank)
            seed_scores[operation_id] += contribution
            seed_reasons[operation_id].append(f"{source_name}@{rank}")

    expanded: dict[str, float] = defaultdict(float)
    reasons: dict[str, list[str]] = defaultdict(list)
    for seed, seed_score in sorted(seed_scores.items()):
        if seed not in graph:
            continue
        for neighbor in sorted(graph.successors(seed)):
            edge_data = graph.get_edge_data(seed, neighbor) or {}
            relation = edge_data.get("relation", "unknown")
            expanded[neighbor] += seed_score * 0.5
            reasons[neighbor].append(f"{seed}:{relation}")

    ranked = sorted(expanded, key=lambda item: (-expanded[item], item))
    return [
        (
            operation_id,
            expanded[operation_id],
            "expanded_from=" + "|".join(sorted(reasons[operation_id])[:8]),
        )
        for operation_id in ranked[: CONFIG["graph_top_k"]]
    ]


def run_candidate_query(query: str) -> list[dict[str, Any]]:
    sources: dict[str, list[tuple[str, float, str]]] = {}

    exact = exact_candidates(query)
    if exact:
        sources["exact"] = exact

    word_scores = (word_matrix @ word_vectorizer.transform([query]).T).toarray().ravel()
    word_hits = [
        (operation_id, score, "word_tfidf")
        for operation_id, score in stable_top_scores(
            word_scores,
            CONFIG["lexical_top_k"],
        )
    ]
    if word_hits:
        sources["word_tfidf"] = word_hits

    char_scores = (char_matrix @ char_vectorizer.transform([query]).T).toarray().ravel()
    char_hits = [
        (operation_id, score, "char_tfidf")
        for operation_id, score in stable_top_scores(
            char_scores,
            CONFIG["lexical_top_k"],
        )
    ]
    if char_hits:
        sources["char_tfidf"] = char_hits

    for plane_id, space in sorted(embedding_spaces.items()):
        query_vector = encode_texts([query], space["model_key"])[0]
        dense_scores = space["matrix"] @ query_vector
        dense_hits = [
            (operation_id, score, f"dense_plane:{plane_id}")
            for operation_id, score in stable_top_scores(
                dense_scores,
                CONFIG["dense_top_k"],
            )
        ]
        if dense_hits:
            sources[f"dense:{plane_id}"] = dense_hits

    graph_hits = graph_candidates(sources)
    if graph_hits:
        sources["graph_expansion"] = graph_hits

    fused_scores: dict[str, float] = defaultdict(float)
    provenance: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for source_name, ranking in sorted(sources.items()):
        for rank, (operation_id, raw_score, reason) in enumerate(ranking, start=1):
            contribution = 1.0 / (CONFIG["rrf_rank_constant"] + rank)
            fused_scores[operation_id] += contribution
            provenance[operation_id].append(
                {
                    "source": source_name,
                    "rank": rank,
                    "raw_score": float(raw_score),
                    "rrf_contribution": contribution,
                    "reason": reason,
                }
            )

    ranked_ids = sorted(
        fused_scores,
        key=lambda operation_id: (-fused_scores[operation_id], operation_id),
    )[: CONFIG["final_top_k"]]
    return [
        {
            "operation_id": operation_id,
            "fused_score": fused_scores[operation_id],
            "provenance": sorted(
                provenance[operation_id],
                key=lambda item: (item["source"], item["rank"]),
            ),
        }
        for operation_id in ranked_ids
    ]


query_rows: list[dict[str, Any]] = []
candidate_rows: list[dict[str, Any]] = []
search_started = time.perf_counter()

for query_index, query in enumerate(CONFIG["queries"]):
    query_id = digest_json({"query": query})
    query_rows.append(
        {
            "query_id": query_id,
            "query_index": query_index,
            "query": query,
            "has_relevance_labels": False,
        }
    )
    results = run_candidate_query(query)
    for rank, result in enumerate(results, start=1):
        operation = operation_by_id[result["operation_id"]]
        candidate_rows.append(
            {
                "query_id": query_id,
                "query": query,
                "rank": rank,
                "operation_id": result["operation_id"],
                "package_name": operation["package_name"],
                "module": operation["module"],
                "qualified_name": operation["qualified_name"],
                "kind": operation["kind"],
                "fused_score": result["fused_score"],
                "source_count": len(result["provenance"]),
                "source_provenance_json": canonical_json(result["provenance"]),
            }
        )

queries_frame = pd.DataFrame(query_rows)
candidates = pd.DataFrame(candidate_rows)
search_seconds = time.perf_counter() - search_started
print(
    canonical_json(
        {
            "queries": len(queries_frame),
            "candidate_rows": len(candidates),
            "search_seconds": round(search_seconds, 6),
            "ground_truth_metrics_emitted": False,
            "reason": "this feature-generation notebook has no relevance or execution labels",
        }
    )
)
candidates.head(20)


## Persist and validate experiment artifacts

The final cell writes immutable run artifacts, appends the new feature partition, validates every JSON provenance cell, reloads Parquet/vector artifacts, and emits a manifest containing actual counts and file digests. It deliberately reports no retrieval-quality metric without ground truth.


In [ ]:
artifact_started = time.perf_counter()

operations_path = RUN_DIR / "operations.parquet"
edges_path = RUN_DIR / "candidate_edges.parquet"
queries_path = RUN_DIR / "queries.parquet"
candidates_path = RUN_DIR / "candidate_results.parquet"
config_path = RUN_DIR / "experiment_config.json"

operations.to_parquet(operations_path, index=False)
edges.to_parquet(edges_path, index=False)
queries_frame.to_parquet(queries_path, index=False)
candidates.to_parquet(candidates_path, index=False)
config_path.write_text(
    json.dumps(
        {
            "config": CONFIG,
            "config_digest": CONFIG_DIGEST,
            "model_registry": MODEL_REGISTRY,
            "bootstrap": BOOTSTRAP_CONFIG,
            "catalog_source": catalog_source,
        },
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

append_path, registry_snapshot_path, registry_snapshot = registry.flush()

# Validate JSON-valued columns and immutable vector references.
for column in [
    "value_json",
    "source_ref_json",
    "evidence_json",
    "generator_config_json",
]:
    for value in registry_snapshot[column].astype(str):
        json.loads(value)
for value in candidates.get(
    "source_provenance_json",
    pd.Series(dtype=str),
).astype(str):
    json.loads(value)
for value in edges.get("evidence_json", pd.Series(dtype=str)).astype(str):
    json.loads(value)

reloaded_operations = pd.read_parquet(operations_path)
reloaded_candidates = pd.read_parquet(candidates_path)
reloaded_registry = pd.read_parquet(registry_snapshot_path)
if len(reloaded_operations) != len(operations):
    raise AssertionError("operation artifact row count changed during round trip")
if len(reloaded_candidates) != len(candidates):
    raise AssertionError("candidate artifact row count changed during round trip")
if reloaded_registry["feature_id"].duplicated().any():
    raise AssertionError("feature registry snapshot contains duplicate feature IDs")

for item in embedding_manifest:
    matrix = np.load(item["artifact"], allow_pickle=False)
    if list(matrix.shape) != item["shape"]:
        raise AssertionError(f"embedding shape changed for {item['plane_id']}")
    if sha256_file(Path(item["artifact"])) != item["sha256"]:
        raise AssertionError(f"embedding digest changed for {item['plane_id']}")

artifact_paths = [
    operations_path,
    edges_path,
    queries_path,
    candidates_path,
    config_path,
    embedding_manifest_path,
    append_path,
    registry_snapshot_path,
    *[Path(item["artifact"]) for item in embedding_manifest],
]
artifact_records = [
    {
        "path": str(path),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]

family_counts = (
    registry_snapshot["feature_kind"].value_counts().sort_index().to_dict()
    if not registry_snapshot.empty
    else {}
)
manifest = {
    "run_id": RUN_ID,
    "started_at_utc": RUN_STARTED.isoformat(),
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "config_digest": CONFIG_DIGEST,
    "catalog_source": catalog_source,
    "safety": {
        "target_packages_imported_or_executed": False,
        "fallback_method": "installed metadata, static source reads, and ast parsing",
        "repository_bootstrap_enabled": BOOTSTRAP_CONFIG["enabled"],
    },
    "counts": {
        "operations": len(operations),
        "catalog_representations_consumed": len(representations),
        "catalog_signals_consumed": len(signals),
        "new_feature_rows": len(registry.new_rows),
        "registry_snapshot_rows": len(registry_snapshot),
        "candidate_edges": len(edges),
        "queries": len(queries_frame),
        "candidate_results": len(candidates),
    },
    "feature_kind_counts": family_counts,
    "embedding_spaces": embedding_manifest,
    "quality_metrics": {
        "emitted": False,
        "reason": "no relevance, accepted-route, or execution labels were supplied",
    },
    "timings_seconds": {
        "catalog_load": round(time.perf_counter() - catalog_started, 6),
        "candidate_search": round(search_seconds, 6),
        "artifact_finalize": round(time.perf_counter() - artifact_started, 6),
        "total": round(time.perf_counter() - catalog_started, 6),
    },
    "artifacts": artifact_records,
}
manifest_path = RUN_DIR / "run_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, sort_keys=True),
    encoding="utf-8",
)
json.loads(manifest_path.read_text(encoding="utf-8"))

print(
    json.dumps(
        {
            "run_directory": str(RUN_DIR.resolve()),
            "manifest": str(manifest_path.resolve()),
            "feature_registry_partition_directory": str(REGISTRY_DIR.resolve()),
            "counts": manifest["counts"],
            "feature_kind_counts": family_counts,
            "quality_metrics": manifest["quality_metrics"],
            "validation": "passed",
        },
        indent=2,
        sort_keys=True,
    )
)


## Next experiment

Attach execution-backed request labels before scoring retrieval quality. The emitted queries and candidates are suitable for inspection and for constructing relevance judgments, multiple acceptable operation sets, hard negatives, no-reuse cases, and route-execution tests. Until those labels exist, candidate counts and timings are diagnostics—not evidence that retrieval is correct.
